In [1]:
import pandas as pd
from pathlib import Path

OUT = Path("../data_processed")
products = pd.read_csv(OUT / "product_facts.csv")
reviews = pd.read_csv(OUT / "reviews.csv")

# 选评论最多的 10 个产品（更容易看出差异）
top10 = (reviews.groupby("product_id").size()
         .sort_values(ascending=False).head(10).index.tolist())
top10

['P420652',
 'P7880',
 'P218700',
 'P248407',
 'P269122',
 'P394639',
 'P450271',
 'P417238',
 'P427421',
 'P411387']

In [2]:
import random, json

def pick_keywords(category, pos_map, row):
    kws = pos_map.get(category, [])
    if kws:
        return kws[:10]
    # 兜底：用 highlights
    highlights = str(row.get("highlights",""))
    kws = [x.strip().strip("'") for x in highlights.strip("[]").split(",") if x.strip()]
    return kws[:10]

# 读 Day3 的关键词候选（如果有）
lex_pos = pd.read_csv(OUT / "keyword_lexicon_pos_candidates.csv")
pos_map = (lex_pos.sort_values(["category","freq"], ascending=[True, False])
           .groupby("category")["keyword"].apply(lambda x: list(x.head(30))).to_dict())

def generate_title_and_bullets(row, keywords, seed):
    random.seed(seed)
    brand = row["brand_name"]
    name = row["product_name"]
    category = row["primary_category"]

    k = [kw for kw in keywords if isinstance(kw,str)]
    random.shuffle(k)
    kw1 = k[0] if len(k)>0 else "daily"
    kw2 = k[1] if len(k)>1 else "light"

    title = f"{brand} {name} | {kw1}"[:60]
    bullets = [
        kw1[:40],
        kw2[:40],
        "absorbs quickly",
        "great for daily routine",
        "patch test if sensitive"
    ]
    return title, bullets

rows = []
for pid in top10:
    row = products.loc[products["product_id"] == pid].iloc[0]
    cat = row["primary_category"]
    kws = pick_keywords(cat, pos_map, row)
    for vid in range(1, 4):  # 1,2,3
        title, bullets = generate_title_and_bullets(row, kws, seed=vid)
        rows.append({
            "product_id": pid,
            "category": cat,
            "variant_id": f"v{vid}",
            "title": title,
            "bullets": "\n".join([f"- {b}" for b in bullets])
        })

pack = pd.DataFrame(rows)
pack.head(6)

,product_id,category,variant_id,title,bullets
0,P420652,Skincare,v1,LANEIGE Lip Sleeping Mask Intense Hydration wi...,- like\n- face\n- absorbs quickly\n- great for...
1,P420652,Skincare,v2,LANEIGE Lip Sleeping Mask Intense Hydration wi...,- feel\n- using\n- absorbs quickly\n- great fo...
2,P420652,Skincare,v3,LANEIGE Lip Sleeping Mask Intense Hydration wi...,- skin\n- feel\n- absorbs quickly\n- great for...
3,P7880,Skincare,v1,fresh Soy Hydrating Gentle Face Cleanser | like,- like\n- face\n- absorbs quickly\n- great for...
4,P7880,Skincare,v2,fresh Soy Hydrating Gentle Face Cleanser | feel,- feel\n- using\n- absorbs quickly\n- great fo...
5,P7880,Skincare,v3,fresh Soy Hydrating Gentle Face Cleanser | skin,- skin\n- feel\n- absorbs quickly\n- great for...


In [3]:
pack_path = OUT / "human_eval_pack.csv"
pack.to_csv(pack_path, index=False)
print("saved:", pack_path)

saved: ../data_processed/human_eval_pack.csv


In [4]:
pack["product_id"].nunique(), pack["variant_id"].nunique()

(10, 3)

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

OUT = Path("../data_processed")

pack = pd.read_csv(OUT / "human_eval_pack.csv")   # 你 Part A 生成的题库
assert set(["product_id","variant_id","title","bullets"]).issubset(pack.columns)

# ---- 可调参数 ----
N_RATERS = 15  # 模拟评审者数量
SEED = 42
np.random.seed(SEED)

# 简单“偏好模型”：更短标题 + bullets里含更多关键词(看起来像卖点) → 更容易被选
def heuristic_score(title, bullets):
    title = str(title)
    bullets = str(bullets).lower()
    # 1) 标题越短越好（更像电商）
    s1 = -len(title) / 60.0
    # 2) bullets里出现这些“购买驱动词”加分（你也可以换成你自己的关键词库）
    drivers = ["hydr", "moist", "clean", "light", "soft", "smooth", "long", "recommend", "gentle"]
    s2 = sum(1 for w in drivers if w in bullets)
    # 3) 少量噪声：模拟人类不一致
    noise = np.random.normal(0, 0.3)
    return 1.2*s2 + 1.0*s1 + noise

# 为每个 product_id 预计算三版本分数
scores = []
for pid, sub in pack.groupby("product_id"):
    sub = sub.sort_values("variant_id")
    for _, r in sub.iterrows():
        scores.append({
            "product_id": pid,
            "variant_id": r["variant_id"],
            "score": heuristic_score(r["title"], r["bullets"])
        })
score_df = pd.DataFrame(scores)

# Softmax 把分数变成“选择概率”
def softmax(x):
    x = np.array(x)
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

# 生成合成选择记录（长表）
rows = []
product_ids = sorted(pack["product_id"].unique().tolist())

for i in range(N_RATERS):
    rater_id = f"synthetic_{i+1:02d}"
    for pid in product_ids:
        sub = score_df[score_df["product_id"] == pid].sort_values("variant_id")
        probs = softmax(sub["score"].values)
        chosen_idx = np.random.choice(len(sub), p=probs)
        chosen_variant = sub.iloc[chosen_idx]["variant_id"]

        rows.append({
            "product_id": pid,
            "variant_id": chosen_variant,
            "chosen": 1,
            "rater_id": rater_id,
            "data_source": "synthetic"   # 明确标注合成
        })

human_pref = pd.DataFrame(rows)
save_path = OUT / "human_preference.csv"
human_pref.to_csv(save_path, index=False)

print("saved:", save_path)
human_pref.head(10)

saved: ../data_processed/human_preference.csv


,product_id,variant_id,chosen,rater_id,data_source
0,P218700,v3,1,synthetic_01,synthetic
1,P248407,v1,1,synthetic_01,synthetic
2,P269122,v1,1,synthetic_01,synthetic
3,P394639,v2,1,synthetic_01,synthetic
4,P411387,v1,1,synthetic_01,synthetic
5,P417238,v3,1,synthetic_01,synthetic
6,P420652,v2,1,synthetic_01,synthetic
7,P427421,v2,1,synthetic_01,synthetic
8,P450271,v1,1,synthetic_01,synthetic
9,P7880,v2,1,synthetic_01,synthetic


In [8]:
# 每个 (product_id, variant_id) 的票数
cnt = human_pref.groupby(["product_id", "variant_id"]).size().reset_index(name="cnt")

# 每个 product_id 的总票数
cnt["total"] = cnt.groupby("product_id")["cnt"].transform("sum")

# 胜率
cnt["win_rate"] = (cnt["cnt"] / cnt["total"]).round(3)

# 看每个产品胜率最高的版本
win_rate = cnt.sort_values(["product_id", "win_rate"], ascending=[True, False])
win_rate.head(15)

,product_id,variant_id,cnt,total,win_rate
0,P218700,v1,8,15,0.533
2,P218700,v3,4,15,0.267
1,P218700,v2,3,15,0.200
3,P248407,v1,8,15,0.533
4,P248407,v2,5,15,0.333
5,P248407,v3,2,15,0.133
6,P269122,v1,8,15,0.533
8,P269122,v3,6,15,0.400
7,P269122,v2,1,15,0.067
9,P394639,v1,8,15,0.533
